# Week 3- Order Analysis with PySpark

# Step 1: Install PySpark

In [1]:
!pip install pyspark

In [2]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, when, datediff, current_date, sum as spark_sum

#Initialize spark
spark = SparkSession.builder.appName("Week3_CustomerOrderAnalysis").getOrCreate()

# Step 2: Upload CSV

In [3]:
from google.colab import files
uploaded = files.upload()

Saving orders.csv to orders.csv
Saving delivery_status.csv to delivery_status.csv
Saving customers.csv to customers.csv


# Step 3: Load CSV Files into Spark DataFrames

In [30]:
orders = spark.read.csv("orders.csv", header=True, inferSchema=True)
delivery_status = spark.read.csv("delivery_status.csv", header=True, inferSchema=True)
customers = spark.read.csv("customers.csv", header=True, inferSchema=True)

# Preview the data
orders.show(3)
delivery_status.show(3)
customers.show(3)



+--------+-----------+----------+-------------+---------+
|order_id|customer_id|order_date|delivery_date|   status|
+--------+-----------+----------+-------------+---------+
|       1|          1|2025-07-01|   2025-07-03|Delivered|
|       2|          2|2025-07-05|   2025-07-08|Delivered|
|       3|          3|2025-07-10|   2025-07-12|  Pending|
+--------+-----------+----------+-------------+---------+
only showing top 3 rows

+-----------+--------+--------------+-------------------+
|delivery_id|order_id|current_status|       last_updated|
+-----------+--------+--------------+-------------------+
|          1|       1|     Delivered|2025-07-03 10:00:00|
|          2|       2|     Delivered|2025-07-08 15:30:00|
|          3|       3|    In Transit|2025-07-20 11:45:00|
+-----------+--------+--------------+-------------------+
only showing top 3 rows

+-----------+------------+-----------------+----------+------+
|customer_id|        name|            email|     phone|region|
+-----------

# Step 4: Join the table

In [31]:
# 2. Join orders + delivery_status + customers tables
joined = orders.join(delivery_status, orders["order_id"] == delivery_status["order_id"]) \
                     .join(customers, orders["customer_id"] == customers["customer_id"])

joined.show()
joined.printSchema()

+--------+-----------+----------+-------------+---------+-----------+--------+--------------+-------------------+-----------+------------+------------------+----------+-------+
|order_id|customer_id|order_date|delivery_date|   status|delivery_id|order_id|current_status|       last_updated|customer_id|        name|             email|     phone| region|
+--------+-----------+----------+-------------+---------+-----------+--------+--------------+-------------------+-----------+------------+------------------+----------+-------+
|       1|          1|2025-07-01|   2025-07-03|Delivered|          1|       1|     Delivered|2025-07-03 10:00:00|          1| Rahul Kumar| rahul@example.com|9876543210|  North|
|       2|          2|2025-07-05|   2025-07-08|Delivered|          2|       2|     Delivered|2025-07-08 15:30:00|          2|Anita Sharma| anita@example.com|9123456780|  South|
|       3|          3|2025-07-10|   2025-07-12|  Pending|          3|       3|    In Transit|2025-07-20 11:45:00|  

# Step 5: Add Delay Column

In [33]:
# Rename order_id in delivery_status before joining
delivery_status = delivery_status.withColumnRenamed("order_id", "delivery_order_id")

# Join
joined = orders.join(delivery_status, orders.order_id == delivery_status.delivery_order_id, "inner") \
                     .join(customers, "customer_id")

# Add the delay column
joined = joined.withColumn(
    "is_delayed",
    when(col("last_updated").cast("date") > col("delivery_date"), 1).otherwise(0)
)

# Select with renamed columns
joined.select("order_id", "delivery_date", "last_updated", "is_delayed").show()


+--------+-------------+-------------------+----------+
|order_id|delivery_date|       last_updated|is_delayed|
+--------+-------------+-------------------+----------+
|       1|   2025-07-03|2025-07-03 10:00:00|         0|
|       2|   2025-07-08|2025-07-08 15:30:00|         0|
|       3|   2025-07-12|2025-07-20 11:45:00|         1|
|       4|   2025-07-15|2025-07-21 09:00:00|         1|
|       5|   2025-07-20|2025-07-23 08:00:00|         1|
|       6|   2025-07-23|2025-07-23 14:20:00|         0|
|       7|   2025-07-25|2025-07-25 16:10:00|         0|
+--------+-------------+-------------------+----------+



In [35]:
#  Extract region
from pyspark.sql.functions import split
joined = joined.withColumn("region", split(col("region"), ",").getItem(0))

joined.select("customer_id", "region").show()

+-----------+-------+
|customer_id| region|
+-----------+-------+
|          1|  North|
|          2|  South|
|          3|   East|
|          4|   West|
|          1|  North|
|          5|Central|
|          6|  South|
+-----------+-------+



#  Step 6: Group by region to count delayed orders

In [36]:
from pyspark.sql.functions import count
delay_by_region = joined.groupBy("region").agg(count(when(col("is_delayed") == 1, True)).alias("delayed_orders"))

# Show result
delay_by_region.show()


+-------+--------------+
| region|delayed_orders|
+-------+--------------+
|  South|             0|
|Central|             0|
|   East|             1|
|   West|             1|
|  North|             1|
+-------+--------------+



# Step 6: Save the Output as CSV

In [38]:
#  Save to CSV
delay_by_region.write.mode("overwrite").csv("output/delayed_orders_by_region", header=True)

# Merge parts into one file
!cat output/delayed_orders_by_region/part-*.csv > delayed_orders_by_region.csv

# 2. Download the Output file showing delayed orders by region
from google.colab import files
files.download("delayed_orders_by_region.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>